# Azzaro: Whisper turbo vs small vs large

Comparamos tres modelos de Whisper sobre el mismo video, contamos diferencias de palabras y revisamos los clips donde no coinciden. El caso puntual a mirar es `racingista` / `racinguista`.

## 1. Configuracion

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    resumen_diferencias,
    normalizar_texto,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None  # si ya tenes el mp4 local, poner aca la ruta y dejar VIDEO_URL como esta

MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"  # poner None si YouTube no pide login
FORCE_TRANSCRIBE = False

MAX_DIFFERENCE_CLIPS = 80
MARGIN_SECONDS = 1.25
PATRON_REVISION = r"racingista|racinguista|racing"

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR

## 2. Descargar o cargar video

In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

## 3. Transcribir con turbo, small y large

Esto cachea cada JSON en `evaluation/outputs/azzaro_whisper/transcripts/`. Si ya existe, no vuelve a transcribir.

In [ ]:
resultados = {}
palabras = {}

for model_name in MODELOS:
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[model_name] = extraer_palabras(resultados[model_name], model_name)

pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras_modelo)}
    for modelo, palabras_modelo in palabras.items()
])

## 4. Alinear palabras y contar diferencias

Se comparan los tres pares: `turbo-small`, `turbo-large` y `small-large`. Despues se exportan clips cortos para revisar manualmente.

In [ ]:
todas_las_diferencias = []
resumenes = []

for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=5,
    )
    resumen = resumen_diferencias(diferencias, palabras[modelo_a], palabras[modelo_b])
    resumen["comparacion"] = f"{modelo_a}_vs_{modelo_b}"
    resumenes.append(resumen)

    for diferencia in diferencias:
        diferencia = dict(diferencia)
        diferencia["comparacion"] = f"{modelo_a}_vs_{modelo_b}"
        diferencia["modelo_a"] = modelo_a
        diferencia["modelo_b"] = modelo_b
        diferencia["texto_a"] = diferencia["turbo_text"]
        diferencia["texto_b"] = diferencia["small_text"]
        diferencia["contexto_a"] = diferencia["contexto_turbo"]
        diferencia["contexto_b"] = diferencia["contexto_small"]
        todas_las_diferencias.append(diferencia)

for diff_id, diferencia in enumerate(todas_las_diferencias, start=1):
    diferencia["diff_id"] = diff_id

df_resumen = pd.DataFrame(resumenes)
display(df_resumen)

diferencias_con_clips = exportar_clips_diferencias(
    video_path,
    todas_las_diferencias,
    output_dir=WORKDIR / "clips_diferencias",
    margen=MARGIN_SECONDS,
    max_clips=MAX_DIFFERENCE_CLIPS,
)

df_clips = pd.DataFrame(diferencias_con_clips)
if "review_index" not in df_clips.columns:
    df_clips.insert(0, "review_index", range(len(df_clips)))

df_resumen.to_csv(WORKDIR / "resumen_turbo_small_large.csv", index=False)
df_clips.to_csv(WORKDIR / "diferencias_con_clips.csv", index=False)

cols = ["review_index", "comparacion", "diff_id", "start", "end", "texto_a", "texto_b", "clip_path"]
display(df_clips[cols])

## 5. Ver casos donde difieren los tres modelos

Esta seccion agrupa diferencias por tramo de video. Se queda con los casos donde `turbo`, `small` y `large` proponen tres textos distintos, y despues muestra cada clip con los tres textos.

In [ ]:
def texto_modelo_en_rango(palabras_modelo, start, end, margen=0.20):
    seleccionadas = []
    left = max(float(start) - margen, 0.0)
    right = float(end) + margen
    for palabra in palabras_modelo:
        p_start = float(palabra.get("start", 0.0))
        p_end = float(palabra.get("end", p_start))
        centro = (p_start + p_end) / 2
        if left <= centro <= right or (p_start < right and p_end > left):
            seleccionadas.append(str(palabra.get("word", "")).strip())
    texto = " ".join(tok for tok in seleccionadas if tok)
    return texto.strip()


def unir_intervalos(intervalos, gap=0.75):
    intervalos = sorted(intervalos, key=lambda x: (x[0], x[1]))
    unidos = []
    for start, end in intervalos:
        if not unidos or start > unidos[-1][1] + gap:
            unidos.append([float(start), float(end)])
        else:
            unidos[-1][1] = max(unidos[-1][1], float(end))
    return unidos


intervalos_diff = [
    (float(d["start"]), float(d["end"]))
    for d in todas_las_diferencias
    if float(d.get("end", 0)) > float(d.get("start", 0))
]

diferencias_tres_modelos = []
for start, end in unir_intervalos(intervalos_diff):
    textos = {
        modelo: texto_modelo_en_rango(palabras[modelo], start, end)
        for modelo in MODELOS
    }
    textos_norm = {modelo: normalizar_texto(texto) for modelo, texto in textos.items()}
    textos_validos = [txt for txt in textos_norm.values() if txt]

    # Queremos casos donde los tres modelos digan algo y las tres versiones sean distintas.
    if len(textos_validos) == 3 and len(set(textos_validos)) == 3:
        diferencias_tres_modelos.append({
            "diff_id": len(diferencias_tres_modelos) + 1,
            "start": round(start, 3),
            "end": round(end, 3),
            "texto_turbo": textos["turbo"],
            "texto_small": textos["small"],
            "texto_large": textos["large"],
            "norm_turbo": textos_norm["turbo"],
            "norm_small": textos_norm["small"],
            "norm_large": textos_norm["large"],
        })

diferencias_tres_modelos = exportar_clips_diferencias(
    video_path,
    diferencias_tres_modelos,
    output_dir=WORKDIR / "clips_tres_modelos",
    margen=MARGIN_SECONDS,
    max_clips=len(diferencias_tres_modelos),
)

df_tres = pd.DataFrame(diferencias_tres_modelos)
if not df_tres.empty and "review_index" not in df_tres.columns:
    df_tres.insert(0, "review_index", range(len(df_tres)))

df_tres.to_csv(WORKDIR / "diferencias_tres_modelos.csv", index=False)
print(f"Casos donde difieren los tres modelos: {len(df_tres)}")
display(df_tres[["review_index", "start", "end", "texto_turbo", "texto_small", "texto_large", "clip_path"]] if not df_tres.empty else df_tres)


In [ ]:
MAX_CASOS_A_MOSTRAR = None  # poner un numero si queres limitar, por ejemplo 10

if df_tres.empty:
    print("No hay casos donde turbo, small y large tengan tres textos distintos.")
else:
    filas = df_tres if MAX_CASOS_A_MOSTRAR is None else df_tres.head(MAX_CASOS_A_MOSTRAR)
    for _, row in filas.iterrows():
        display(Markdown(f"### Caso {int(row['review_index'])} | {row['start']}s - {row['end']}s"))
        print("turbo:", row["texto_turbo"])
        print("small:", row["texto_small"])
        print("large:", row["texto_large"])
        display(Video(row["clip_path"], embed=True, html_attributes="controls"))


In [ ]:
# Opcional: guardar una decision manual para un caso puntual.
REVIEW_INDEX = 0
MODELO_CORRECTO = ""  # opciones: "turbo", "small", "large", "ninguno", "empate"
NOTA = ""

revision_path = WORKDIR / "revision_manual_tres_modelos.csv"

if df_tres.empty:
    print("No hay casos de tres modelos para guardar.")
else:
    if revision_path.exists():
        revision_tres = pd.read_csv(revision_path)
    else:
        revision_tres = df_tres.copy()
        revision_tres["modelo_correcto"] = ""
        revision_tres["nota"] = ""

    if MODELO_CORRECTO:
        mask = revision_tres["review_index"].eq(REVIEW_INDEX)
        revision_tres.loc[mask, "modelo_correcto"] = MODELO_CORRECTO
        revision_tres.loc[mask, "nota"] = NOTA
        revision_tres.to_csv(revision_path, index=False)
        display(revision_tres.loc[mask, ["review_index", "texto_turbo", "texto_small", "texto_large", "modelo_correcto", "nota"]])
        print(f"Guardado en: {revision_path}")
    else:
        completadas = revision_tres["modelo_correcto"].fillna("").ne("").sum()
        print(f"Completa MODELO_CORRECTO para guardar. Revisadas: {completadas} / {len(revision_tres)}")
        display(revision_tres[["review_index", "texto_turbo", "texto_small", "texto_large", "modelo_correcto", "nota"]])
